# 九曜：天墟 EP01 — Wan 2.2 TI2V-5B 影片試作（Kaggle 免費 GPU）

- 目標：用 Wan 2.2 TI2V-5B（Apache 2.0，可商用、無浮水印）把 E01-01 的分鏡圖做成影片，和可靈版比較。
- **Kaggle 設定**（右側 Session options）：Accelerator = **GPU T4 x2**（只用 1 張）、Internet = **On**（需先完成手機驗證）。
- 由上而下依序執行。第 1 次先跑預設的「快速測試」（480×832、3 秒），確認能跑、看耗時，再把 `QUALITY` 改成 `"final"` 跑 720p。
- 首幀只能用 1 張圖（TI2V-5B 不支援尾幀），所以 E01-01 的兩鏡分開生成：鏡 1 用 f1、鏡 2 用 f2，剪輯時接起來。
- 成品在右側 Output（`/kaggle/working/*.mp4`），下載後交給 Claude 驗片、進版控。

In [ ]:
# 1. 安裝套件（約 1～2 分鐘）
!pip install -q -U "diffusers>=0.35" transformers accelerate ftfy imageio-ffmpeg sentencepiece
import os
os.environ["HF_HOME"] = "/tmp/hf"          # 模型約 20GB+，放 /tmp（/kaggle/working 只有 20GB）
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. 參數（要改的都在這裡）
QUALITY = "test"   # "test" = 480×832、30 步（先確認跑得動）；"final" = 704×1280、50 步
SEED = 42
FPS = 24           # TI2V-5B 原生 24fps
NUM_FRAMES = 73    # 必須是 4k+1；73 幀 ≈ 3 秒（分鏡規格：兩鏡各 3 秒）

W, H, STEPS = (480, 832, 30) if QUALITY == "test" else (704, 1280, 50)
GUIDANCE = 5.0
MODEL_ID = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
RAW = "https://raw.githubusercontent.com/sky03104/jiuyao-tianxu/main/production/EP01/shuohao/storyboard/export/h3/E01-01"

# 提示詞改寫自 storyboard/export/h3/E01-01/prompt.md，每鏡各自生成（單鏡、無切鏡、無對白）
STYLE = ("Eastern fantasy anime-style illustration, painterly semi-realistic rendering, soft morning side light, "
         "consistent with the reference frame, no people, no text, no subtitles.")
SHOTS = [
    {"name": "E01-01_shot1", "image": f"{RAW}/f1.png",
     "prompt": "An extreme wide view of a vast mountain academy climbing the ridge in three tiers, a band of white cloud "
               "drifting slowly across the middle tier; the camera makes a slow, steady push in toward the triple-eaved "
               "main hall at the top while distant deep-blue banners stir in the morning wind. " + STYLE},
    {"name": "E01-01_shot2", "image": f"{RAW}/f2.png",
     "prompt": "Low-angle view of the bronze-framed gatehouse at the foot of the mountain, centered composition, static "
               "camera; two deep-blue banners sway gently in the morning breeze and mist drifts slowly behind the gate, "
               "stone lions and lanterns stay still. " + STYLE},
]
NEGATIVE = ("色调艳丽，过曝，静态，细节模糊不清，字幕，文字，水印，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，"
            "JPEG压缩残留，丑陋的，残缺的，畸形的，人物，行人，杂乱的背景，建筑变形，镜头抖动")
print(QUALITY, W, H, STEPS, NUM_FRAMES)

In [ ]:
# 3. 先在 CPU 上把提示詞編碼完，再把文字編碼器丟掉
#    （T4 不支援 bf16 運算、UMT5 用 fp16 容易溢位變 NaN，所以文字編碼器在 CPU 上以 bf16 跑，只跑一次，約數分鐘）
import torch, gc, time
from diffusers import WanImageToVideoPipeline

t0 = time.time()
enc = WanImageToVideoPipeline.from_pretrained(MODEL_ID, vae=None, transformer=None, torch_dtype=torch.bfloat16)
EMBEDS = {}
with torch.no_grad():
    for s in SHOTS:
        pe, ne = enc.encode_prompt(s["prompt"], NEGATIVE, do_classifier_free_guidance=True,
                                   max_sequence_length=512, device="cpu", dtype=torch.bfloat16)
        EMBEDS[s["name"]] = (pe, ne)
del enc; gc.collect()
print(f"prompt 編碼完成 {time.time()-t0:.0f}s")

In [ ]:
# 4. 載入影片模型：transformer fp16 上 GPU（自動 CPU offload），VAE 用 fp32＋tiling 避免解碼爆顯存
from diffusers import AutoencoderKLWan
DTYPE = torch.float16   # 若輸出全黑／NaN，見第 6 格說明
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanImageToVideoPipeline.from_pretrained(MODEL_ID, vae=vae, text_encoder=None, torch_dtype=DTYPE)
pipe.vae.enable_tiling()
pipe.enable_model_cpu_offload(device="cuda:0")

In [ ]:
# 5. 生成：分鏡圖（1024×1536，2:3）置中裁成目標比例再縮放，當首幀
import numpy as np
from diffusers.utils import load_image, export_to_video

def fit(img, w, h):
    iw, ih = img.size; r = w / h
    if iw / ih > r: nw = int(ih * r); img = img.crop(((iw - nw) // 2, 0, (iw - nw) // 2 + nw, ih))
    else: nh = int(iw / r); img = img.crop((0, (ih - nh) // 2, iw, (ih - nh) // 2 + nh))
    return img.resize((w, h))

RESULTS = []
for s in SHOTS:
    pe, ne = EMBEDS[s["name"]]
    img = fit(load_image(s["image"]).convert("RGB"), W, H)
    t0 = time.time()
    frames = pipe(image=img, prompt_embeds=pe.to(DTYPE), negative_prompt_embeds=ne.to(DTYPE),
                  height=H, width=W, num_frames=NUM_FRAMES, num_inference_steps=STEPS, guidance_scale=GUIDANCE,
                  generator=torch.Generator("cpu").manual_seed(SEED)).frames[0]
    out = f"/kaggle/working/{s['name']}_wan22_{QUALITY}_{W}x{H}.mp4"
    export_to_video(frames, out, fps=FPS)
    bad = bool(np.isnan(frames).any() or frames.max() < 0.02)
    RESULTS.append((out, time.time() - t0, bad))
    print(f"{out}  {time.time()-t0:.0f}s  {'⚠️ 全黑/NaN' if bad else 'OK'}  峰值顯存 {torch.cuda.max_memory_allocated()/2**30:.1f}GB")

### 6. 結果回報給 Claude
把下一格印出的內容連同 mp4 一起交給 Claude。

**常見狀況**
- **全黑／NaN**：fp16 溢位。把第 4 格 `DTYPE` 改成 `torch.bfloat16` 重跑第 4、5 格（T4 會用模擬運算，較慢但數值穩定）。
- **CUDA OOM**：`QUALITY="final"` 時可能發生，先把 `NUM_FRAMES` 降到 49（約 2 秒），或第 4 格把
  `enable_model_cpu_offload` 換成 `pipe.enable_sequential_cpu_offload(device="cuda:0")`（很慢但省顯存）。
- **系統記憶體不足（kernel 重啟）**：Kaggle 約 29GB RAM；重開 session 從第 1 格重跑即可（第 3 格結束已釋放文字編碼器）。

In [ ]:
import platform, diffusers, transformers
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| diffusers", diffusers.__version__)
print("QUALITY", QUALITY, W, H, "steps", STEPS, "frames", NUM_FRAMES, "dtype", DTYPE)
for out, sec, bad in RESULTS: print(out, f"{sec/60:.1f} 分鐘", "BAD" if bad else "OK")